In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')  # resolve backend modules from notebooks/

## 1. Config & registry

In [2]:
from app.config import get_settings
from clients.provider_registry import REGISTRY

settings = get_settings()
print(f"Default provider : {settings.default_provider}")
print(f"Default model    : {settings.default_model}")
print(f"OpenAI key set   : {bool(settings.openai_api_key)}")
print(f"Anthropic key set: {bool(settings.anthropic_api_key)}")
print()
for name, info in REGISTRY.items():
    print(f"{name}: {info.models}")

Default provider : openai
Default model    : gpt-4o-mini
OpenAI key set   : True
Anthropic key set: True

openai: ('gpt-4o', 'gpt-4o-mini', 'gpt-4-turbo', 'o1-mini')
anthropic: ('claude-opus-4-7', 'claude-sonnet-4-6', 'claude-3-5-haiku-latest')
google: ('gemini-1.5-pro', 'gemini-1.5-flash', 'gemini-2.0-flash')


## 2. Model factory — no API call

In [3]:
from pydantic_ai.models.test import TestModel
from pydantic_ai import Agent
from models.outputs import NewsAnalysis
from prompts.news import SYSTEM

agent = Agent(TestModel(), output_type=NewsAnalysis, system_prompt=SYSTEM)
result = await agent.run("Ticker: AAPL\n\nHeadlines:\n- Apple beats Q2 earnings expectations")

print(type(result.output))
print(result.output.model_dump_json(indent=2))

<class 'models.outputs.NewsAnalysis'>
{
  "ticker": "a",
  "sentiment": "bullish",
  "summary": "a",
  "key_risks": [],
  "key_opportunities": [],
  "confidence": 0.0
}


## 3. Live call — OpenAI

In [5]:
from clients.llm import build_model

model = build_model(provider="openai", model_name="gpt-4o-mini")
agent = Agent(model, output_type=NewsAnalysis, system_prompt=SYSTEM)

result = await agent.run(
    "Ticker: NVDA\n\nHeadlines:\n"
    "- Nvidia Blackwell chips see record demand\n"
    "- China export curbs may cut $15B in revenue\n"
    "- Data center revenue up 427% YoY"
)

out = result.output
print(f"Ticker      : {out.ticker}")
print(f"Sentiment   : {out.sentiment}")
print(f"Confidence  : {out.confidence:.0%}")
print(f"\nSummary:\n{out.summary}")
print(f"\nRisks:")
for r in out.key_risks:
    print(f"  - {r}")
print(f"\nOpportunities:")
for o in out.key_opportunities:
    print(f"  - {o}")

Ticker      : NVDA
Sentiment   : bullish
Confidence  : 80%

Summary:
Nvidia is experiencing strong demand for its Blackwell chips and a significant increase in data center revenue, indicating robust growth potential. However, potential disruptions from China's export restrictions pose a risk to future revenue streams.

Risks:
  - China export curbs leading to a potential $15B revenue loss

Opportunities:
  - Record demand for Blackwell chips
  - 427% YoY growth in data center revenue


## 4. Live call — Anthropic

In [ ]:
model = build_model(provider="anthropic", model_name="claude-3-5-haiku-latest")
agent = Agent(model, output_type=NewsAnalysis, system_prompt=SYSTEM)

result = await agent.run(
    "Ticker: TSLA\n\nHeadlines:\n"
    "- Tesla misses Q1 deliveries, down 13% YoY\n"
    "- Cybertruck recall issued for third time\n"
    "- Musk hints at affordable model launch in H2"
)

print(result.output.model_dump_json(indent=2))

## 5. Usage & cost metadata

In [6]:
print(result.usage())

RunUsage(input_tokens=201, output_tokens=221, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=1)
